In [9]:
import torch
import torchvision
import torchvision.transforms as transforms

In [10]:
#data augmentation + tensor conversion + normalisation(in the range of -1 and 1)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip() , 
    transforms.RandomCrop(32 , padding = 4),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5 , 0.5 , 0.5) , 
        (0.5 , 0.5 , 0.5)
    )
])

In [11]:
transform_test = transforms.Compose([

    transforms.ToTensor() , 
    transforms.Normalize(
        (0.5 , 0.5 , 0.5),
        (0.5 , 0.5 , 0.5)
    )

])

In [14]:
trainset = torchvision.datasets.CIFAR100(
    root = "/datasets",
    train = True ,
    download = True , 
    transform = transform_train

)

100.0%
c:\Users\ELCOT\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [15]:
testset = torchvision.datasets.CIFAR100(
    root = "/datasets",
    train = False ,
    download = True , 
    transform = transform_test

)

In [16]:
train_loader = torch.utils.data.DataLoader(trainset , batch_size=64 , shuffle=True)
test_loader = torch.utils.data.DataLoader(testset , batch_size = 64 , shuffle = False)

In [21]:
import torch.nn as nn
import torch.nn.functional as F

class CIFAR100CNN(nn.Module):

    def __init__(self):
        super().__init__()

        
        self.conv1 = nn.Conv2d(3 , 32 , kernel_size=3 , stride = 1 , padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32 , 64 , kernel_size=3 , stride=1 , padding = 1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64 , 128 , kernel_size=3 , stride=1 , padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2 , 2)

        self.fc1 = nn.Linear(128 * 4 * 4 , 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 100)

    def forward(self , x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x





In [22]:
model = CIFAR100CNN()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [23]:
for epoch in range(10):
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images, labels

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print("Epoch:", epoch, "Loss:", running_loss)

Epoch: 0 Loss: 3129.1801755428314
Epoch: 1 Loss: 2788.309770345688
Epoch: 2 Loss: 2636.217858314514
Epoch: 3 Loss: 2528.733055114746
Epoch: 4 Loss: 2438.3222250938416
Epoch: 5 Loss: 2369.136674642563
Epoch: 6 Loss: 2312.298168182373
Epoch: 7 Loss: 2262.8332302570343
Epoch: 8 Loss: 2211.283308506012
Epoch: 9 Loss: 2171.039636850357


In [26]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images, labels

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 39.47
